In [ ]:

-- =====================================================
-- CELL 2: POPULATE MART.DIM_DATE
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

MERGE INTO MART.DIM_DATE AS T
USING (
    WITH RECURSIVE

    SOURCE_DATES AS (
        SELECT ENROLLMENT_DATE AS BUSINESS_DATE
        FROM CORE.MEMBER_PROFILE
        WHERE ENROLLMENT_DATE IS NOT NULL

        UNION ALL

        SELECT LAST_FLIGHT_DATE
        FROM CORE.MEMBER_PROFILE
        WHERE LAST_FLIGHT_DATE IS NOT NULL

        UNION ALL

        SELECT FEED_DATE
        FROM CORE.REDEMPTION_TRANSACTIONS
        WHERE FEED_DATE IS NOT NULL

        UNION ALL

        SELECT TXN_DATE
        FROM CORE.REDEMPTION_TRANSACTIONS
        WHERE TXN_DATE IS NOT NULL
    ),

    DATE_BOUNDS AS (
        SELECT
            MIN(BUSINESS_DATE) AS MIN_DATE,
            MAX(BUSINESS_DATE) AS MAX_DATE
        FROM SOURCE_DATES
    ),

    DATE_SPINE (FULL_DATE) AS (
        SELECT MIN_DATE
        FROM DATE_BOUNDS
        WHERE MIN_DATE IS NOT NULL

        UNION ALL

        SELECT DATEADD(DAY, 1, D.FULL_DATE)::DATE
        FROM DATE_SPINE D
        CROSS JOIN DATE_BOUNDS B
        WHERE D.FULL_DATE < B.MAX_DATE
    )

    SELECT
        TO_NUMBER(TO_CHAR(FULL_DATE, 'YYYYMMDD')) AS DATE_KEY,
        FULL_DATE,
        DAYOFMONTH(FULL_DATE) AS DAY_NUMBER,
        DAYNAME(FULL_DATE) AS DAY_NAME,
        WEEKISO(FULL_DATE) AS WEEK_NUMBER,
        MONTH(FULL_DATE) AS MONTH_NUMBER,
        MONTHNAME(FULL_DATE) AS MONTH_NAME,
        QUARTER(FULL_DATE) AS QUARTER_NUMBER,
        YEAR(FULL_DATE) AS YEAR_NUMBER,
        DAYOFWEEKISO(FULL_DATE) IN (6, 7) AS IS_WEEKEND
    FROM DATE_SPINE

) AS S
ON T.FULL_DATE = S.FULL_DATE

WHEN MATCHED THEN UPDATE SET
    T.DATE_KEY = S.DATE_KEY,
    T.DAY_NUMBER = S.DAY_NUMBER,
    T.DAY_NAME = S.DAY_NAME,
    T.WEEK_NUMBER = S.WEEK_NUMBER,
    T.MONTH_NUMBER = S.MONTH_NUMBER,
    T.MONTH_NAME = S.MONTH_NAME,
    T.QUARTER_NUMBER = S.QUARTER_NUMBER,
    T.YEAR_NUMBER = S.YEAR_NUMBER,
    T.IS_WEEKEND = S.IS_WEEKEND

WHEN NOT MATCHED THEN INSERT (
    DATE_KEY,
    FULL_DATE,
    DAY_NUMBER,
    DAY_NAME,
    WEEK_NUMBER,
    MONTH_NUMBER,
    MONTH_NAME,
    QUARTER_NUMBER,
    YEAR_NUMBER,
    IS_WEEKEND
)
VALUES (
    S.DATE_KEY,
    S.FULL_DATE,
    S.DAY_NUMBER,
    S.DAY_NAME,
    S.WEEK_NUMBER,
    S.MONTH_NUMBER,
    S.MONTH_NAME,
    S.QUARTER_NUMBER,
    S.YEAR_NUMBER,
    S.IS_WEEKEND
);

-- VERIFY DATE DIMENSION

SELECT
    COUNT(*) AS TOTAL_DATES,
    MIN(FULL_DATE) AS FIRST_DATE,
    MAX(FULL_DATE) AS LAST_DATE,
    COUNT(DISTINCT DATE_KEY) AS UNIQUE_DATE_KEYS
FROM MART.DIM_DATE;

SELECT *
FROM MART.DIM_DATE
ORDER BY FULL_DATE
LIMIT 20;


In [ ]:

-- =====================================================
-- CELL 3: POPULATE MART.DIM_MEMBER
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

MERGE INTO MART.DIM_MEMBER AS T
USING (
    SELECT
        MEMBER_ID,
        MEMBER_NAME,
        ENROLLMENT_DATE,
        LAST_FLIGHT_DATE,
        TIER_CODE,
        STATE,
        COUNTRY,
        AGE,
        ACTIVE_MEMBER,
        STALE_MEMBER,
        SOURCE_FILE_NAME,
        FILE_TIMESTAMP
    FROM CORE.MEMBER_PROFILE
    WHERE MEMBER_ID IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MEMBER_ID
        ORDER BY
            FILE_TIMESTAMP DESC NULLS LAST,
            RECORD_UPDATED_TIMESTAMP DESC NULLS LAST,
            INGESTION_TIMESTAMP DESC NULLS LAST
    ) = 1
) AS S
ON T.MEMBER_ID = S.MEMBER_ID

WHEN MATCHED THEN UPDATE SET
    T.MEMBER_NAME = S.MEMBER_NAME,
    T.ENROLLMENT_DATE = S.ENROLLMENT_DATE,
    T.LAST_FLIGHT_DATE = S.LAST_FLIGHT_DATE,
    T.TIER_CODE = S.TIER_CODE,
    T.STATE = S.STATE,
    T.COUNTRY = S.COUNTRY,
    T.AGE = S.AGE,
    T.ACTIVE_MEMBER = S.ACTIVE_MEMBER,
    T.STALE_MEMBER = S.STALE_MEMBER,
    T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME,
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP

WHEN NOT MATCHED THEN INSERT (
    MEMBER_KEY,
    MEMBER_ID,
    MEMBER_NAME,
    ENROLLMENT_DATE,
    LAST_FLIGHT_DATE,
    TIER_CODE,
    STATE,
    COUNTRY,
    AGE,
    ACTIVE_MEMBER,
    STALE_MEMBER,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP
)
VALUES (
    SKYPOINTS_INCU.MART.SEQ_MEMBER_KEY.NEXTVAL,
    S.MEMBER_ID,
    S.MEMBER_NAME,
    S.ENROLLMENT_DATE,
    S.LAST_FLIGHT_DATE,
    S.TIER_CODE,
    S.STATE,
    S.COUNTRY,
    S.AGE,
    S.ACTIVE_MEMBER,
    S.STALE_MEMBER,
    S.SOURCE_FILE_NAME,
    S.FILE_TIMESTAMP
);

-- VERIFY MEMBER DIMENSION

SELECT
    COUNT(*) AS TOTAL_MEMBERS,
    COUNT(DISTINCT MEMBER_ID) AS UNIQUE_MEMBERS
FROM MART.DIM_MEMBER;

SELECT
    MEMBER_KEY,
    MEMBER_ID,
    MEMBER_NAME,
    COUNTRY,
    TIER_CODE,
    AGE
FROM MART.DIM_MEMBER
ORDER BY MEMBER_ID;


In [ ]:

-- =====================================================
-- CELL 4: POPULATE MART.FACT_REDEMPTION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

MERGE INTO MART.FACT_REDEMPTION AS T
USING (
    SELECT
        D.DATE_KEY,
        M.MEMBER_KEY,
        R.MEMBER_ID,
        R.TXN_ID,
        R.TXN_DATE,
        R.PARTNER,
        R.MILES_REDEEMED,
        R.STATUS,
        R.SOURCE_FILE_NAME,
        R.BATCH_ID
    FROM CORE.REDEMPTION_TRANSACTIONS R

    INNER JOIN MART.DIM_MEMBER M
        ON R.MEMBER_ID = M.MEMBER_ID

    INNER JOIN MART.DIM_DATE D
        ON R.TXN_DATE = D.FULL_DATE

    WHERE R.MEMBER_ID IS NOT NULL
      AND R.TXN_ID IS NOT NULL
      AND R.TXN_DATE IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY R.MEMBER_ID, R.TXN_ID
        ORDER BY
            R.FILE_TIMESTAMP DESC NULLS LAST,
            R.RECORD_UPDATED_TIMESTAMP DESC NULLS LAST,
            R.INGESTION_TIMESTAMP DESC NULLS LAST
    ) = 1

) AS S

ON T.MEMBER_ID = S.MEMBER_ID
AND T.TXN_ID = S.TXN_ID

WHEN MATCHED THEN UPDATE SET
    T.DATE_KEY = S.DATE_KEY,
    T.MEMBER_KEY = S.MEMBER_KEY,
    T.TXN_DATE = S.TXN_DATE,
    T.PARTNER = S.PARTNER,
    T.MILES_REDEEMED = S.MILES_REDEEMED,
    T.STATUS = S.STATUS,
    T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME,
    T.BATCH_ID = S.BATCH_ID

WHEN NOT MATCHED THEN INSERT (
    REDEMPTION_KEY,
    DATE_KEY,
    MEMBER_KEY,
    MEMBER_ID,
    TXN_ID,
    TXN_DATE,
    PARTNER,
    MILES_REDEEMED,
    STATUS,
    SOURCE_FILE_NAME,
    BATCH_ID
)
VALUES (
    ROW_NUMBER() OVER (ORDER BY S.MEMBER_ID, S.TXN_ID),
    S.DATE_KEY,
    S.MEMBER_KEY,
    S.MEMBER_ID,
    S.TXN_ID,
    S.TXN_DATE,
    S.PARTNER,
    S.MILES_REDEEMED,
    S.STATUS,
    S.SOURCE_FILE_NAME,
    S.BATCH_ID
);


In [ ]:
SHOW SEQUENCES IN SCHEMA MART;

In [ ]:
%%sql -r dataframe_8
-- KEEP THIS CELL: Creates the surrogate key sequence for FACT_REDEMPTION.
-- IF NOT EXISTS makes this safe to execute again.

USE DATABASE SKYPOINTS_INCU;
USE SCHEMA MART;

CREATE SEQUENCE IF NOT EXISTS SEQ_REDEMPTION_KEY
    START = 1
    INCREMENT = 1;